<a href="https://colab.research.google.com/github/KruthikaVirupakshappa/DATA266-Lab1-Pair37/blob/jiayi-work/task2_sentiment/Task2_team_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
from pathlib import Path
import pandas as pd

task2_root = Path("task2_sentiment")
prediction_paths = list(task2_root.glob("*/outputs/test_predictions.csv"))

assert len(prediction_paths) >= 2, "Expected at least two test_predictions.csv files."

def get_columns(path):
    return pd.read_csv(path, nrows=0).columns.tolist()

pred_set_a_path = next(
    p for p in prediction_paths
    if any(col.endswith("_meanmax") for col in get_columns(p))
)

pred_set_b_path = next(
    p for p in prediction_paths
    if "prob_exp1_bilstm" in get_columns(p)
)

evaluate_path = pred_set_b_path.parents[1] / "src" / "evaluate.py"

print("Mean+Max / MLP / Attention predictions:", pred_set_a_path.exists())
print("Mean Embedding / BiLSTM / TextCNN predictions:", pred_set_b_path.exists())
print("Shared evaluate.py:", evaluate_path.exists())


Mean+Max / MLP / Attention predictions: True
Mean Embedding / BiLSTM / TextCNN predictions: True
Shared evaluate.py: True


In [26]:
import importlib.util
import numpy as np

spec = importlib.util.spec_from_file_location("task2_evaluate", evaluate_path)
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)


In [27]:
pred_set_a = pd.read_csv(pred_set_a_path)
pred_set_b = pd.read_csv(pred_set_b_path)

print("Mean+Max / MLP / Attention:", pred_set_a.shape)
print("Mean Embedding / BiLSTM / TextCNN:", pred_set_b.shape)

display(pred_set_a.head())
display(pred_set_b.head())


Mean+Max / MLP / Attention: (38000, 5)
Mean Embedding / BiLSTM / TextCNN: (38000, 8)


,test_index,true,prob_meanmax,prob_mlp,prob_attention
0,0,1,0.419917,0.236557,0.129016
1,1,0,0.569177,0.414076,0.445325
2,2,1,0.965561,0.879681,0.836962
3,3,0,0.091126,0.092837,0.272902
4,4,1,0.683702,0.624933,0.727146


,test_index,true,length,negation,contrast,prob_baseline_mean_embedding,prob_exp1_bilstm,prob_exp2_textcnn
0,0,1,medium (51-200),has negation,has contrast word,0.317011,0.238763,0.127074
1,1,0,medium (51-200),has negation,no contrast word,0.153102,0.008597,0.000190
2,2,1,short (<=50 words),no negation,no contrast word,0.999376,0.968213,0.875913
3,3,0,medium (51-200),no negation,no contrast word,0.026684,0.018535,0.109611
4,4,1,medium (51-200),has negation,no contrast word,0.735054,0.889221,0.875084


In [28]:
assert len(pred_set_a) == 38000
assert len(pred_set_b) == 38000
assert np.array_equal(pred_set_a["test_index"].to_numpy(), pred_set_b["test_index"].to_numpy())
assert np.array_equal(pred_set_a["true"].to_numpy(), pred_set_b["true"].to_numpy())

print("PASS: both prediction files contain the same 38,000 test examples in the same order.")


PASS: both prediction files contain the same 38,000 test examples in the same order.


In [29]:
y = pred_set_a["true"].to_numpy().astype(int)

def prob_by_suffix(df, suffix):
    matches = [col for col in df.columns if col.endswith(suffix)]
    assert len(matches) == 1, f"Expected one probability column ending with {suffix}, found {matches}"
    return df[matches[0]].to_numpy()

probs = {
    "Mean+Max": prob_by_suffix(pred_set_a, "_meanmax"),
    "MLP": prob_by_suffix(pred_set_a, "_mlp"),
    "Attention": prob_by_suffix(pred_set_a, "_attention"),
    "Mean Embedding": pred_set_b["prob_baseline_mean_embedding"].to_numpy(),
    "BiLSTM": pred_set_b["prob_exp1_bilstm"].to_numpy(),
    "TextCNN": pred_set_b["prob_exp2_textcnn"].to_numpy()
}

for model_name, p in probs.items():
    assert len(p) == len(y)
    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()
    print(model_name, "min =", p.min(), "max =", p.max())


Mean+Max min = 3.570164e-19 max = 1.0
MLP min = 7.7274855e-17 max = 1.0
Attention min = 4.069044e-11 max = 1.0
Mean Embedding min = 0.0 max = 1.0
BiLSTM min = 0.0 max = 1.0
TextCNN min = 0.0 max = 1.0


In [30]:
slice_labels = {
    "length": pred_set_b["length"].to_numpy(),
    "negation": pred_set_b["negation"].to_numpy(),
    "contrast": pred_set_b["contrast"].to_numpy()
}

for slice_name, labels in slice_labels.items():
    print("\n", slice_name)
    print(pd.Series(labels).value_counts())



 length
medium (51-200)       21172
short (<=50 words)     9362
long (>200)            7466
Name: count, dtype: int64

 negation
has negation    28177
no negation      9823
Name: count, dtype: int64

 contrast
has contrast word    22075
no contrast word     15925
Name: count, dtype: int64


In [31]:
shared_metric_rows = []

for model_name, p in probs.items():
    metrics = ev.classification_metrics(y, p, ece_bins=15)
    metrics.update(ev.bootstrap_ci(y, p, n_boot=1000, seed=0))

    shared_metric_rows.append({
        "model": model_name,
        **metrics
    })

shared_metrics = pd.DataFrame(shared_metric_rows)

display(shared_metrics)

,model,accuracy,precision_macro,recall_macro,f1_macro,precision_micro,recall_micro,f1_micro,precision_weighted,recall_weighted,...,mcc,brier,ece,error_rate,accuracy_ci_low,accuracy_ci_high,f1_macro_ci_low,f1_macro_ci_high,mcc_ci_low,mcc_ci_high
0,Mean+Max,0.927158,0.927179,0.927158,0.927157,0.927158,0.927158,0.927158,0.927179,0.927158,...,0.854337,0.053845,0.007401,0.072842,0.924605,0.929658,0.924603,0.929656,0.849242,0.859337
1,MLP,0.928289,0.928527,0.928289,0.928280,0.928289,0.928289,0.928289,0.928527,0.928289,...,0.856816,0.053315,0.008344,0.071711,0.925656,0.930948,0.925647,0.930934,0.851493,0.862077
2,Attention,0.926132,0.926201,0.926132,0.926129,0.926132,0.926132,0.926132,0.926201,0.926132,...,0.852333,0.054570,0.004551,0.073868,0.923474,0.928712,0.923473,0.928711,0.847042,0.857546
3,Mean Embedding,0.932605,0.932636,0.932605,0.932604,0.932605,0.932605,0.932605,0.932636,0.932605,...,0.865241,0.051699,0.011801,0.067395,0.930026,0.935159,0.930026,0.935157,0.860102,0.870316
4,BiLSTM,0.953105,0.953118,0.953105,0.953105,0.953105,0.953105,0.953105,0.953118,0.953105,...,0.906223,0.035943,0.013495,0.046895,0.951052,0.955211,0.951052,0.955206,0.902128,0.910431
5,TextCNN,0.943526,0.943554,0.943526,0.943525,0.943526,0.943526,0.943526,0.943554,0.943526,...,0.887080,0.042398,0.004882,0.056474,0.941158,0.945947,0.941158,0.945939,0.882350,0.891888


In [32]:
comparison_cols = [
    "model",
    "accuracy",
    "accuracy_ci_low",
    "accuracy_ci_high",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "f1_macro_ci_low",
    "f1_macro_ci_high",
    "f1_micro",
    "f1_weighted",
    "roc_auc",
    "pr_auc",
    "mcc",
    "mcc_ci_low",
    "mcc_ci_high",
    "brier",
    "ece"
]

display(shared_metrics[comparison_cols].round(4))

,model,accuracy,accuracy_ci_low,accuracy_ci_high,precision_macro,recall_macro,f1_macro,f1_macro_ci_low,f1_macro_ci_high,f1_micro,f1_weighted,roc_auc,pr_auc,mcc,mcc_ci_low,mcc_ci_high,brier,ece
0,Mean+Max,0.9272,0.9246,0.9297,0.9272,0.9272,0.9272,0.9246,0.9297,0.9272,0.9272,0.9789,0.9789,0.8543,0.8492,0.8593,0.0538,0.0074
1,MLP,0.9283,0.9257,0.9309,0.9285,0.9283,0.9283,0.9256,0.9309,0.9283,0.9283,0.9796,0.9797,0.8568,0.8515,0.8621,0.0533,0.0083
2,Attention,0.9261,0.9235,0.9287,0.9262,0.9261,0.9261,0.9235,0.9287,0.9261,0.9261,0.9790,0.9794,0.8523,0.8470,0.8575,0.0546,0.0046
3,Mean Embedding,0.9326,0.9300,0.9352,0.9326,0.9326,0.9326,0.9300,0.9352,0.9326,0.9326,0.9790,0.9788,0.8652,0.8601,0.8703,0.0517,0.0118
4,BiLSTM,0.9531,0.9511,0.9552,0.9531,0.9531,0.9531,0.9511,0.9552,0.9531,0.9531,0.9902,0.9902,0.9062,0.9021,0.9104,0.0359,0.0135
5,TextCNN,0.9435,0.9412,0.9459,0.9436,0.9435,0.9435,0.9412,0.9459,0.9435,0.9435,0.9867,0.9871,0.8871,0.8824,0.8919,0.0424,0.0049


In [33]:
slice_rows = []

for model_name, p in probs.items():
    rows = ev.slice_metrics(y, p, slice_labels)

    for row in rows:
        slice_rows.append({
            "model": model_name,
            **row
        })

shared_slices = pd.DataFrame(slice_rows)

display(shared_slices)

,model,slice,value,n,f1_macro,error_rate
0,Mean+Max,length,long (>200),7466,0.919422,0.075810
1,Mean+Max,length,medium (51-200),21172,0.929860,0.070140
2,Mean+Max,length,short (<=50 words),9362,0.920142,0.076586
3,Mean+Max,negation,has negation,28177,0.919244,0.078220
4,Mean+Max,negation,no negation,9823,0.921086,0.057416
5,Mean+Max,contrast,has contrast word,22075,0.917795,0.081495
6,Mean+Max,contrast,no contrast word,15925,0.937984,0.060848
7,MLP,length,long (>200),7466,0.921414,0.073399
8,MLP,length,medium (51-200),21172,0.929989,0.069998
9,MLP,length,short (<=50 words),9362,0.923047,0.074236


In [34]:
slice_comparison = shared_slices.pivot_table(
    index=["slice", "value"],
    columns="model",
    values=["f1_macro", "error_rate"]
)

display(slice_comparison.round(4))

error_rate                                 \
model                        Attention  BiLSTM     MLP Mean Embedding   
slice    value                                                          
contrast has contrast word      0.0843  0.0511  0.0804         0.0751   
         no contrast word       0.0595  0.0410  0.0597         0.0568   
length   long (>200)            0.0781  0.0438  0.0734         0.0650   
         medium (51-200)        0.0725  0.0453  0.0700         0.0677   
         short (<=50 words)     0.0735  0.0530  0.0742         0.0687   
negation has negation           0.0803  0.0481  0.0773         0.0725   
         no negation            0.0554  0.0435  0.0557         0.0528   

                                              f1_macro                  \
model                       Mean+Max TextCNN Attention  BiLSTM     MLP   
slice    value                                                           
contrast has contrast word    0.0815  0.0613    0.9151  0.9484  0.9186   
         no contrast word     0.0608  0.0498    0.9394  0.9582  0.9393   
length   long (>200)          0.0758  0.0571    0.9174  0.9535  0.9214   
         medium (51-200)      0.0701  0.0546    0.9274  0.9547  0.9300   
         short (<=50 words)   0.0766  0.0604    0.9234  0.9448  0.9230   
negation has negation         0.0782  0.0577    0.9172  0.9504  0.9197   
         no negation          0.0574  0.0528    0.9237  0.9407  0.9249   

                                                             
model                       Mean Embedding Mean+Max TextCNN  
slice    value                                               
contrast has contrast word          0.9241   0.9178  0.9382  
         no contrast word           0.9422   0.9380  0.9492  
length   long (>200)                0.9305   0.9194  0.9394  
         medium (51-200)            0.9323   0.9299  0.9454  
         short (<=50 words)         0.9285   0.9201  0.9370  
negation has negation               0.9249   0.9192  0.9404  
         no negation                0.9274   0.9211  0.9272

In [35]:
mcnemar_pairs = [
    ("Mean+Max", "MLP"),
    ("Mean+Max", "Attention"),
    ("Mean Embedding", "BiLSTM"),
    ("Mean Embedding", "TextCNN")
]

mcnemar_rows = []

for baseline_name, experimental_name in mcnemar_pairs:
    result = ev.mcnemar(y, probs[baseline_name], probs[experimental_name])

    mcnemar_rows.append({
        "baseline": baseline_name,
        "experimental": experimental_name,
        **result
    })

shared_mcnemar = pd.DataFrame(mcnemar_rows)

display(shared_mcnemar)


,baseline,experimental,b_base_only_correct,c_exp_only_correct,chi2,p_value,p_value_exact
0,Mean+Max,MLP,538,581,1.576408,2.092789e-01,2.092594e-01
1,Mean+Max,Attention,784,745,0.944408,3.311468e-01,3.311482e-01
2,Mean Embedding,BiLSTM,588,1367,309.608184,2.658149e-69,3.366031e-71
3,Mean Embedding,TextCNN,677,1092,96.888638,7.333834e-23,4.666639e-23


In [36]:
shared_output_dir = Path("task2_sentiment/shared_outputs")
shared_output_dir.mkdir(parents=True, exist_ok=True)

for old_plot in shared_output_dir.glob("*_confusion_matrix.png"):
    old_plot.unlink()


In [37]:
for model_name, p in probs.items():
    safe_name = model_name.lower().replace("+", "plus").replace(" ", "_")
    ev.plot_confusion(y, p, model_name, shared_output_dir / f"{safe_name}_confusion_matrix.png")

In [38]:
ev.plot_curves(y, probs, shared_output_dir / "six_model_roc_pr_calibration.png")

In [39]:
shared_metrics.to_csv(shared_output_dir / "six_model_metrics.csv", index=False)
shared_slices.to_csv(shared_output_dir / "six_model_slice_metrics.csv", index=False)
shared_mcnemar.to_csv(shared_output_dir / "mcnemar.csv", index=False)

print("Saved shared evaluation to:", shared_output_dir)

Saved shared evaluation to: task2_sentiment/shared_outputs


In [40]:
model_metadata = pd.DataFrame([
    {"model": "Mean+Max", "parameters": 3840257, "train_time_s": 533.6522, "train_examples_per_s": 4722.1766, "peak_gpu_mem_mb": 89.9038, "best_epoch": 4, "batch_size": 64, "cpu": "Intel(R) Xeon(R) CPU @ 2.20GHz", "gpu": "NVIDIA A100-SXM4-40GB"},
    {"model": "MLP", "parameters": 3850369, "train_time_s": 612.5822, "train_examples_per_s": 4113.7335, "peak_gpu_mem_mb": 90.7891, "best_epoch": 4, "batch_size": 64, "cpu": "Intel(R) Xeon(R) CPU @ 2.20GHz", "gpu": "NVIDIA A100-SXM4-40GB"},
    {"model": "Attention", "parameters": 3892417, "train_time_s": 586.0575, "train_examples_per_s": 4299.9192, "peak_gpu_mem_mb": 110.7480, "best_epoch": 3, "batch_size": 64, "cpu": "Intel(R) Xeon(R) CPU @ 2.20GHz", "gpu": "NVIDIA A100-SXM4-40GB"},
    {"model": "Mean Embedding", "parameters": 5140481, "train_time_s": 25.4585, "train_examples_per_s": 98984.6114, "peak_gpu_mem_mb": 965.1680, "best_epoch": 3, "batch_size": 256, "cpu": "not recorded", "gpu": "NVIDIA A100-SXM4-40GB"},
    {"model": "BiLSTM", "parameters": 5405057, "train_time_s": 459.7312, "train_examples_per_s": 5481.4638, "peak_gpu_mem_mb": 1471.2280, "best_epoch": 5, "batch_size": 256, "cpu": "not recorded", "gpu": "NVIDIA A100-SXM4-40GB"},
    {"model": "TextCNN", "parameters": 5337729, "train_time_s": 53.1083, "train_examples_per_s": 47450.1940, "peak_gpu_mem_mb": 1223.8682, "best_epoch": 4, "batch_size": 256, "cpu": "not recorded", "gpu": "NVIDIA A100-SXM4-40GB"}
])

full_comparison = shared_metrics.merge(model_metadata, on="model", how="left")
full_comparison.to_csv(shared_output_dir / "six_model_full_comparison.csv", index=False)

display(full_comparison)


,model,accuracy,precision_macro,recall_macro,f1_macro,precision_micro,recall_micro,f1_micro,precision_weighted,recall_weighted,...,mcc_ci_low,mcc_ci_high,parameters,train_time_s,train_examples_per_s,peak_gpu_mem_mb,best_epoch,batch_size,cpu,gpu
0,Mean+Max,0.927158,0.927179,0.927158,0.927157,0.927158,0.927158,0.927158,0.927179,0.927158,...,0.849242,0.859337,3840257,533.6522,4722.1766,89.9038,4,64,Intel(R) Xeon(R) CPU @ 2.20GHz,NVIDIA A100-SXM4-40GB
1,MLP,0.928289,0.928527,0.928289,0.928280,0.928289,0.928289,0.928289,0.928527,0.928289,...,0.851493,0.862077,3850369,612.5822,4113.7335,90.7891,4,64,Intel(R) Xeon(R) CPU @ 2.20GHz,NVIDIA A100-SXM4-40GB
2,Attention,0.926132,0.926201,0.926132,0.926129,0.926132,0.926132,0.926132,0.926201,0.926132,...,0.847042,0.857546,3892417,586.0575,4299.9192,110.7480,3,64,Intel(R) Xeon(R) CPU @ 2.20GHz,NVIDIA A100-SXM4-40GB
3,Mean Embedding,0.932605,0.932636,0.932605,0.932604,0.932605,0.932605,0.932605,0.932636,0.932605,...,0.860102,0.870316,5140481,25.4585,98984.6114,965.1680,3,256,not recorded,NVIDIA A100-SXM4-40GB
4,BiLSTM,0.953105,0.953118,0.953105,0.953105,0.953105,0.953105,0.953105,0.953118,0.953105,...,0.902128,0.910431,5405057,459.7312,5481.4638,1471.2280,5,256,not recorded,NVIDIA A100-SXM4-40GB
5,TextCNN,0.943526,0.943554,0.943526,0.943525,0.943526,0.943526,0.943526,0.943554,0.943526,...,0.882350,0.891888,5337729,53.1083,47450.1940,1223.8682,4,256,not recorded,NVIDIA A100-SXM4-40GB
